# Data Collection

## Business Purpose

Basket CDS pricing requires market data from multiple reference entities.

In this project, five major U.S. banks are selected as reference names:

- JPMorgan Chase (JPM)
- Bank of America (BAC)
- Citigroup (C)
- Goldman Sachs (GS)
- Morgan Stanley (MS)

Historical stock prices are collected as a proxy for estimating credit dependence among reference entities.

## Objective

The objective of this notebook is to:

1. Collect historical market data
2. Validate data quality
3. Store cleaned data for future analysis

## Step 1: Import Libraries

### Purpose

This step imports the Python libraries required for data collection and analysis.

- yfinance: download historical market data
- pandas: data manipulation
- numpy: numerical calculations

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

## Step 2: Select Reference Entities

### Purpose

Basket CDS pricing requires multiple reference entities.

Five large U.S. financial institutions are selected because:

- They are systemically important banks
- Historical market data is readily available
- Their credit risk is economically related

In [3]:
banks = [
    "JPM",   # JPMorgan Chase
    "BAC",   # Bank of America
    "C",     # Citigroup
    "GS",    # Goldman Sachs
    "MS"     # Morgan Stanley
]

banks

['JPM', 'BAC', 'C', 'GS', 'MS']

## Step 3: Download Historical Market Data

### Purpose

Historical stock prices are used as a proxy for measuring co-movement among institutions.

The data will later be used to estimate:

- Pearson correlation
- Rank correlation
- Default dependence structure

In [4]:
prices = yf.download(
    banks,
    start="2023-01-01",
    end="2025-01-01",
    auto_adjust=True
)

prices.head()

[*********************100%***********************]  5 of 5 completed


Price           Close                                                \
Ticker            BAC          C          GS         JPM         MS   
Date                                                                  
2023-01-03  30.640951  40.644218  318.011658  123.735184  76.617462   
2023-01-04  31.217007  41.691849  319.371033  124.889015  77.448700   
2023-01-05  31.153006  41.505402  315.752106  124.861351  76.796204   
2023-01-06  31.463892  42.002575  319.720123  127.250641  78.262062   
2023-01-09  30.988417  42.206779  324.239258  126.724823  78.333557   

Price            High                                                ...  \
Ticker            BAC          C          GS         JPM         MS  ...   
Date                                                                 ...   
2023-01-03  31.171295  41.505401  320.197751  125.218697  77.404013  ...   
2023-01-04  31.838787  42.206784  321.768369  126.079470  78.074365  ...   
2023-01-05  31.217012  41.691842  317.019674  125.193454  77.019656  ...   
2023-01-06  31.619339  42.339943  320.565179  127.656546  78.619588  ...   
2023-01-09  31.783928  42.899275  327.454095  128.117818  79.468700  ...   

Price            Open                                                  Volume  \
Ticker            BAC          C          GS         JPM         MS       BAC   
Date                                                                            
2023-01-03  30.384925  40.617585  317.350319  123.845082  76.545956  35221500   
2023-01-04  30.997557  41.318968  319.674134  124.531876  76.465510  41998500   
2023-01-05  31.015847  41.461011  316.202174  125.147326  76.778330  34177000   
2023-01-06  31.171290  41.780621  319.003675  125.580905  77.260998  34068700   
2023-01-09  31.774786  42.517514  323.210515  127.859517  78.878782  43818800   

Price                                             
Ticker             C       GS       JPM       MS  
Date                                              
2023-01-03  19564000  1589700  11054800  5108900  
2023-01-04  21507200  1881000  11687600  7726800  
2023-01-05  12314900  1397800   8381300  5339900  
2023-01-06  16736300  3097800  10029100  5710900  
2023-01-09  16745500  1989000   8482300  5039700  

[5 rows x 25 columns]

## Step 4 Extract Adjusted Closing Prices

The downloaded dataset contains multiple variables, including Open, High, Low, Close, and Volume.

For correlation estimation, only adjusted closing prices are required.

These prices will later be converted into daily returns.

In [5]:
close_prices = prices["Close"]

close_prices.head()

Ticker,BAC,C,GS,JPM,MS
Date,,,,,
2023-01-03,30.640951,40.644218,318.011658,123.735184,76.617462
2023-01-04,31.217007,41.691849,319.371033,124.889015,77.448700
2023-01-05,31.153006,41.505402,315.752106,124.861351,76.796204
2023-01-06,31.463892,42.002575,319.720123,127.250641,78.262062
2023-01-09,30.988417,42.206779,324.239258,126.724823,78.333557


In [6]:
plt.figure(figsize=(12,6))

for bank in close_prices.columns:
    plt.plot(close_prices.index, close_prices[bank], label=bank)

plt.title("Adjusted Closing Prices (2020–2025)")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## Step 5 Data Quality Assessment

Before performing any quantitative analysis, the dataset is checked for:

- missing observations,
- dataset dimensions,
- data completeness.

High-quality data is essential because missing values may bias correlation estimates and Monte Carlo simulations.

In [ ]:
print("Dataset Shape:")
print(close_prices.shape)

print("\nMissing Values:")
print(close_prices.isna().sum())

Dataset Shape:
(502, 5)

Missing Values:
Ticker
BAC    0
C      0
GS     0
JPM    0
MS     0
dtype: int64


## Step 6 Save Processed Dataset

The cleaned adjusted closing prices are saved locally.

Saving intermediate datasets improves reproducibility and allows subsequent notebooks to load the same data without repeating the download process.

In [ ]:
close_prices.to_csv("../Data/bank_prices.csv")

## Step 7 Verify Saved Dataset

The saved CSV file is reloaded to verify that the export was successful.

In [ ]:
saved_data = pd.read_csv("../Data/bank_prices.csv")

saved_data.head()

,Date,BAC,C,GS,JPM,MS
0,2023-01-03,30.640949,40.644218,318.011597,123.735207,76.617455
1,2023-01-04,31.217012,41.691841,319.371124,124.889008,77.448700
2,2023-01-05,31.152998,41.505405,315.752106,124.861366,76.796211
3,2023-01-06,31.463890,42.002571,319.720154,127.250633,78.262054
4,2023-01-09,30.988411,42.206779,324.239227,126.724785,78.333572


# Summary

In this notebook, historical adjusted closing prices for five major U.S. banks were collected and saved for future analysis.

Completed tasks:

- Imported required libraries
- Selected five reference entities
- Downloaded historical market data
- Extracted adjusted closing prices
- Performed basic data quality checks
- Saved the processed dataset

The next notebook will estimate default probabilities and hazard rates, providing the foundation for the Basket CDS pricing model.

# Business Interpretation

Historical price data provide the foundation for estimating return distributions and dependence among the reference entities.

Although stock prices are not direct measures of credit risk, equity returns contain valuable information about common market movements and are frequently used to estimate default correlation in reduced-form credit risk models.